# **Kitchen Usage Workflow**

The kitchen monitoring module integrated into DepaYT combines contextual analysis and computer vision techniques to support cleanliness tracking in shared apartment environments.

**1. Start Kitchen Usage**: The workflow begins when a user presses the `Start Usage` button inside the application before using the kitchen. At this stage, the system starts a cooking session associated with the current user.

 **2. Stop Kitchen Usage**: After finishing the cooking activity, the user presses the `Stop Usage` button. Once the session ends, the application displays a contextual questionnaire related to the cooking activity performed.

The questionnaire includes information such as:

- type of food prepared,
- cooking method,
- oil usage,
- spill occurrence,
- number of utensils used,
- number of people served.

**3. Contextual Dirtiness Prediction**: The answers provided by the user are converted into a structured tabular input and processed by a Multilayer Perceptron (MLP) model trained on contextual cooking data.

The MLP predicts the expected kitchen dirtiness level, classified into one of the following categories:

- low,
- medium,
- high.

Based on the predicted level, the system generates personalized cleaning recommendations to guide the user in properly cleaning the kitchen after use.

 **4. Upload Final Kitchen Image**: After receiving the recommendation, the user uploads a final image of the induction kitchen surface. The image is processed by a Convolutional Neural Network (CNN) trained for binary kitchen cleanliness classification.

The CNN predicts whether the kitchen surface is:
- clean,
- dirty.

**5. Save and Share Kitchen Status**
The final cleanliness result, along with the contextual session information, is stored inside the application database.
This information can later be visualized by all roommates, allowing users to monitor kitchen usage history, cleanliness status, and shared kitchen conditions over time.

The proposed workflow combines:
- contextual reasoning through the MLP model,
- visual cleanliness verification through the CNN model,
- collaborative monitoring inside a shared apartment environment.

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
from tensorflow.keras.models import load_model
import tensorflow as tf
from tensorflow.keras.preprocessing import image



In [28]:
preprocessor = joblib.load(
    "Text_Questions/data/preprocessor.joblib"
)

label_encoder = joblib.load(
    "Text_Questions/data/label_encoder.joblib"
)

# Contextual MLP
mlp_model = load_model(
    r"Text_Questions/results/kitchen_context_mlp.keras"
)

# Kitchen CNN
cnn_model = load_model(
    r"Kitchen_Image_Prediction/models/cnn_augmented.keras"
)


In [29]:
# IMAGE PREDICTION FUNCTION
def predict_kitchen_image(image_path, target_size=(224, 224)):
    """
    Predicts kitchen dirtiness from image.
    Returns:
        visual_score
        visual_label
    """

    # LOAD IMAGE
    img = image.load_img(
        image_path,
        target_size=target_size
    )

    # IMAGE TO ARRAY
    img_array = image.img_to_array(img)

    # NORMALIZE
    img_array = img_array / 255.0

    # ADD BATCH DIMENSION
    img_array = np.expand_dims(img_array, axis=0)

    # PREDICT
    prediction = cnn_model.predict(img_array)[0][0]

    # VISUAL SCORE
    visual_score = float(prediction)

    # LABEL
    visual_label = (
        "dirty"
        if visual_score >= 0.5
        else "clean"
    )

    return {
        "visual_score":
            visual_score,
        "visual_label":
            visual_label
    }

In [30]:
# CONTEXTUAL PREDICTION FUNCTION
def predict_contextual_dirtiness(
    user_input_df
):
    """
    Predicts contextual dirtiness
    using questionnaire information.

    Returns:
        expected_level
        probabilities
        context_score
    """

    # COPY INPUT
    user_input_df = user_input_df.copy()

    # BINARY MAPPING
    binary_mapping = {"si": 1, "no": 0}
    binary_columns = ["uso_aceite", "hubo_derrame"]

    for col in binary_columns:
        user_input_df[col] = (
            user_input_df[col]
            .map(binary_mapping)
        )

    # PREPROCESS
    X_user = preprocessor.transform(user_input_df)

    # PREDICT
    y_probs = mlp_model.predict(X_user)
    y_pred = np.argmax(y_probs, axis=1)

    # DECODE LABEL
    expected_level = (
        label_encoder
        .inverse_transform(y_pred)[0]
    )

    # CONTEXT SCORE

    score_mapping = {

        "bajo": 0.33,
        "medio": 0.66,
        "alto": 1.0
    }

    context_score = 0

    for idx, prob in enumerate(y_probs[0]):

        class_name = (
            label_encoder.classes_[idx]
        )

        class_score = (
            score_mapping[class_name]
        )

        context_score += (
            prob * class_score
        )

    return {

        "expected_level":
            expected_level,

        "probabilities":
            y_probs[0],

        "context_score":
            float(context_score)
    }

In [31]:
# RECOMMENDATION GENERATOR
def generate_recommendation(
    final_score,
    user_input_df
):
    """
    Generates contextual recommendation
    using multimodal information.
    """

    oil_used = (
        user_input_df[
            "uso_aceite"
        ].iloc[0] == "si"
    )

    spill_detected = (
        user_input_df[
            "hubo_derrame"
        ].iloc[0] == "si"
    )

    cooking_method = (
        user_input_df[
            "metodo_coccion"
        ].iloc[0]
    )

    # LOW DIRTINESS
    if final_score < 0.40:

        recommendation = (
            "Se detectó un nivel bajo de suciedad en la cocina. "
            "Se recomienda realizar una limpieza ligera de la superficie de inducción utilizando un trapo húmedo o papel absorbente. "
            "Verifica que no existan pequeñas gotas, residuos de comida o utensilios olvidados antes de finalizar la sesión."
        )

    # MEDIUM DIRTINESS
    elif final_score < 0.75:

        recommendation = (
            "Se detectó un nivel moderado de suciedad en la cocina. "
            "Se recomienda limpiar residuos visibles, manchas de aceite y restos de alimentos presentes sobre la superficie de cocción. "
        )

        if oil_used:

            recommendation += (
                "Debido al uso de aceite durante la preparación, se recomienda utilizar productos desengrasantes suaves o papel absorbente para eliminar residuos grasosos. "
            )

        if spill_detected:

            recommendation += (
                "También se detectó posibilidad de derrames, por lo que es importante limpiar líquidos acumulados para evitar manchas o superficies pegajosas. "
            )

    # HIGH DIRTINESS
    else:

        recommendation = (
            "Se detectó un nivel alto de suciedad en la cocina. "
            "La actividad realizada probablemente generó acumulación considerable de grasa, residuos de alimentos o derrames sobre la superficie de inducción. "
            "Se recomienda realizar una limpieza profunda de toda el área utilizada antes de finalizar la sesión. "
        )

        if oil_used:

            recommendation += (
                "Debido al uso de aceite o frituras, se recomienda utilizar productos desengrasantes y un trapo húmedo para remover residuos adheridos. "
            )

        if spill_detected:

            recommendation += (
                "Los derrames detectados deben limpiarse inmediatamente utilizando papel absorbente o paños secos para evitar manchas persistentes o acumulación de suciedad. "
            )

        if cooking_method in ["frito", "salteado"]:
            recommendation += (
                "Las técnicas de cocción utilizadas suelen generar salpicaduras de grasa, por lo que se recomienda revisar también los bordes y zonas cercanas a la cocina. "
            )

        recommendation += (
            "Recuerda dejar el espacio limpio y organizado para el siguiente usuario."
        )

    return recommendation


In [ ]:
def predict_multimodal_kitchen_state(
    user_input_df,
    image_path,
    alpha=0.5
):
    """
    Multimodal kitchen cleanliness prediction.

    Combines:
    - contextual reasoning (MLP)
    - visual reasoning (CNN)

    Returns:
        multimodal prediction results
    """
    
    # CONTEXTUAL PREDICTION
    contextual_result = (predict_contextual_dirtiness(user_input_df))
    context_score = (contextual_result["context_score"])
           
    # IMAGE PREDICTION
    visual_result = (predict_kitchen_image(image_path))
    visual_score = (visual_result["visual_score"])

    # MULTIMODAL FUSION
    final_score = (alpha * visual_score +  (1 - alpha) * context_score)

    if final_score < 0.40:
        final_level = "bajo"

    elif final_score < 0.75:
        final_level = "medio"

    else:
        final_level = "alto"

    # RECOMMENDATION
    recommendation = (
        generate_recommendation(
            final_score,
            user_input_df
        )
    )

    return {
        "final_level": final_level,
        "final_score": float(final_score),
            
        "context_score": float(context_score),
        "visual_score": float(visual_score),
            
        "contextual_prediction": contextual_result["expected_level"],   
        "visual_prediction": visual_result["visual_label"],
            
        "recommendation": recommendation      
    }

# **Sample Data to test recomendations**

In [33]:
example_input = pd.DataFrame([{
    "tipo_comida": "pollo",
    "metodo_coccion": "frito",
    "uso_aceite": "si",
    "hubo_derrame": "si",
    "cantidad_utensilios": "muchos",
    "cantidad_personas": "3_o_mas"
}])

# RUN MULTIMODAL PREDICTION
result = predict_multimodal_kitchen_state(
    user_input_df=example_input,
    image_path="Sample_Images/Dirty1.jpeg",
    alpha=0.5
)


print("\n========== MULTIMODAL RESULT ==========\n")
print(
    f"\nContext Score: "
    f"{result['context_score']:.4f}"
    f"\nContextual Prediction: "
    f"{result['contextual_prediction']}"
)
print(
    f"\nVisual Score: "
    f"{result['visual_score']:.4f}"
    f"\nVisual Prediction: "
    f"{result['visual_prediction']}"
)

print("=====================================")
print(
    f"Final Dirtiness Level: "
    f"{result['final_level']},"
    f"\nFinal Score: "
    f"{result['final_score']:.4f} "
)

print(
    f"\nRecommendation:\n"
    f"{result['recommendation']}"
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step

========== MULTIMODAL RESULT ==========


Context Score: 1.0000
Contextual Prediction: alto

Visual Score: 1.0000
Visual Prediction: dirty
Final Dirtiness Level: alto,
Final Score: 1.0000 

Recommendation:
Se detectó un nivel alto de suciedad en la cocina. La actividad realizada probablemente generó acumulación considerable de grasa, residuos de alimentos o derrames sobre la superficie de inducción. Se recomienda realizar una limpieza profunda de toda el área utilizada antes de finalizar la sesión. Debido al uso de aceite o frituras, se recomienda utilizar productos desengrasantes y un trapo húmedo para remover residuos adheridos. Los derrames detectados deben limpiarse inmediatamente utilizando papel absorbente o paños secos para evitar manchas persistentes o acumulación de suciedad. Las técnicas de cocción utilizadas suelen generar salpicaduras de grasa, por lo que se recomienda revisar también los bordes y 

In [35]:
example_input = pd.DataFrame([{
    "tipo_comida": "agua",
    "metodo_coccion": "hervido",
    "uso_aceite": "no",
    "hubo_derrame": "no",
    "cantidad_utensilios": "pocos",
    "cantidad_personas": "1_o_2"
}])


# RUN MULTIMODAL PREDICTION
result = predict_multimodal_kitchen_state(

    user_input_df=example_input,

    image_path="Sample_Images/clean20.jpg",

    alpha=0.5
)


print("\n========== MULTIMODAL RESULT ==========\n")

print(
    f"\nContext Score: "
    f"{result['context_score']:.4f}"
    
    f"\nContextual Prediction: "
    f"{result['contextual_prediction']}"
)

print(
    f"\nVisual Score: "
    f"{result['visual_score']:.4f}"
    
    f"\nVisual Prediction: "
    f"{result['visual_prediction']}"
)

print("=====================================")

print(
    f"Final Dirtiness Level: "
    f"{result['final_level']},"
    
    f"\nFinal Score: "
    f"{result['final_score']:.4f}"
)

print(
    f"\nRecommendation:\n"
    f"{result['recommendation']}"
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step

========== MULTIMODAL RESULT ==========


Context Score: 0.3300
Contextual Prediction: bajo

Visual Score: 0.0085
Visual Prediction: clean
Final Dirtiness Level: bajo,
Final Score: 0.1693

Recommendation:
Se detectó un nivel bajo de suciedad en la cocina. Se recomienda realizar una limpieza ligera de la superficie de inducción utilizando un trapo húmedo o papel absorbente. Verifica que no existan pequeñas gotas, residuos de comida o utensilios olvidados antes de finalizar la sesión.
